# Hugging Face to Spark NLP: GGUF Conversion Guide

This notebook walks you through converting any compatible Hugging Face model to a Spark NLP-compatible format using GGUF.
What this covers:
- Converting a Hugging Face model to GGUF format using llama.cpp
- Preparing model assets and quantization options
- Loading the exported model into Spark NLP
- Running an end-to-end inference pipeline

# llama.cpp setup/conversion

In [ ]:
# @title basic cpu build: https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md
!git clone https://github.com/ggml-org/llama.cpp
%cd llama.cpp
!cmake -B build
!cmake --build build --config Release -j 4
%cd /content


In [ ]:
# @title hf --> gguf conversion function
# This code cell provides a complete end-to-end pipeline for converting a Hugging Face
# model into GGUF format using llama.cpp, with optional quantization and vision MMProj support.

# What this code does:
# 1. Accepts either a Hugging Face repo ID (for example "org/model") or a local model path.
# 2. Automatically downloads the model from Hugging Face if a repo ID is provided.
# 3. Validates that required llama.cpp tools are available:
#    - convert_hf_to_gguf.py
#    - llama-quantize
# 4. Converts the model to a BF16 GGUF base file.
# 5. Quantizes the base GGUF into one or more formats (for example q4_k_m, q8_0).
# 6. Detects whether the model is vision-capable and safely generates MMProj GGUF files
#    only when supported.

# Example Usage:
# ```
# report = convert_model_to_gguf(
#     source_model="meta-llama/Llama-3.1-8B-Instruct",
#     model_name="Llama-3.1-8B-Instruct",
# )
# ```

# > MMProj Q8 generation is disabled by default and must be explicitly enabled.

import sys, os, json, logging, subprocess, contextlib, time
from typing import List, Dict, Optional
from huggingface_hub import snapshot_download
from transformers import AutoConfig

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s', force=True)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

def validate_path(p, d):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{d} not found at: {p}")

def run_command(cmd, desc, env=None, dry_run=False):
    logging.info(desc)
    logging.info(" ".join(cmd))
    if dry_run:
        return True
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)
        return True
    except subprocess.CalledProcessError as e:
        logging.error(desc)
        if e.stdout: logging.error(e.stdout)
        if e.stderr: logging.error(e.stderr)
        return False

def download_model(model_id, cache_dir="hf_models"):
    logging.info(f"Starting download of model: {model_id}")
    start = time.time()
    path = os.path.join(cache_dir, model_id.replace("/", "__"))
    os.makedirs(path, exist_ok=True)
    snapshot_download(
        repo_id=model_id,
        local_dir=path,
        local_dir_use_symlinks=False,
        tqdm_class=None
    )
    elapsed = time.time() - start
    logging.info(f"Finished download of model: {model_id} in {elapsed:.1f}s")
    return path

def resolve_model_path(src):
    return src if os.path.isdir(src) else download_model(src)

def is_vision_model(path):
    try:
        c = AutoConfig.from_pretrained(path)
        return hasattr(c, "vision_config") or hasattr(c, "mm_vision_tower")
    except Exception:
        return False

def file_exists_and_nonempty(path: str) -> bool:
    return os.path.exists(path) and os.path.getsize(path) > 0

def convert_model_to_gguf(
    source_model: str,
    model_name: str,
    quant_methods: List[str] = ["q4_k_m", "q8_0"],
    output_dir: str = "ggufs",
    threads: Optional[int] = None,
    convert_mmproj: bool = True,
    enable_mmproj_q8: bool = False,
    dry_run: bool = False
) -> Dict:

    report = {"model": model_name, "base": None, "quantized": {}, "mmproj": {}, "success": False}

    try:
        convert_script = "llama.cpp/convert_hf_to_gguf.py"
        quant_bin = "llama.cpp/build/bin/llama-quantize"
        validate_path(convert_script, "convert_hf_to_gguf.py")
        validate_path(quant_bin, "llama-quantize")

        model_path = resolve_model_path(source_model)
        os.makedirs(output_dir, exist_ok=True)

        env = os.environ.copy()
        cpu_threads = os.cpu_count() or 1
        env["OMP_NUM_THREADS"] = str(threads or max(cpu_threads - 1, 1))

        # Base BF16
        base = os.path.join(output_dir, f"{model_name.lower()}.bf16.gguf")
        logging.info("Starting BF16 conversion...")
        start = time.time()
        if not run_command(
            ["python", convert_script, model_path, "--outtype", "bf16", "--outfile", base],
            "BF16 conversion", env, dry_run
        ):
            return report
        elapsed = time.time() - start
        logging.info(f"BF16 conversion completed in {elapsed:.1f}s")

        if not file_exists_and_nonempty(base):
            logging.warning(f"Base GGUF file missing or empty: {base}")
        report["base"] = base

        # Quantization
        for m in map(str.lower, quant_methods):
            out = os.path.join(output_dir, f"{model_name.lower()}.{m}.gguf")
            logging.info(f"Starting quantization: {m}")
            start = time.time()
            success = run_command([quant_bin, base, out, m], f"Quantize {m}", env, dry_run)
            elapsed = time.time() - start
            logging.info(f"Quantization {m} completed in {elapsed:.1f}s")
            if success and not file_exists_and_nonempty(out):
                logging.warning(f"Quantized GGUF file missing or empty: {out}")
                success = False
            report["quantized"][m] = out if success else None

        # MMProj
        if convert_mmproj and is_vision_model(model_path):
            mm_bf16 = os.path.join(output_dir, f"{model_name.lower()}-mmproj.bf16.gguf")
            logging.info("Starting MMProj BF16 conversion...")
            start = time.time()
            if run_command(
                ["python", convert_script, model_path, "--mmproj", "--outtype", "bf16", "--outfile", mm_bf16],
                "MMProj BF16", env, dry_run
            ):
                if not file_exists_and_nonempty(mm_bf16):
                    logging.warning(f"MMProj BF16 file missing or empty: {mm_bf16}")
                else:
                    report["mmproj"]["bf16"] = mm_bf16
            logging.info(f"MMProj BF16 completed in {time.time()-start:.1f}s")

            if enable_mmproj_q8:
                mm_q8 = os.path.join(output_dir, f"{model_name.lower()}-mmproj.q8_0.gguf")
                logging.info("Starting MMProj Q8 conversion...")
                start = time.time()
                if run_command(
                    ["python", convert_script, model_path, "--mmproj", "--outtype", "q8_0", "--outfile", mm_q8],
                    "MMProj Q8_0", env, dry_run
                ):
                    if not file_exists_and_nonempty(mm_q8):
                        logging.warning(f"MMProj Q8 file missing or empty: {mm_q8}")
                    else:
                        report["mmproj"]["q8_0"] = mm_q8
                logging.info(f"MMProj Q8 completed in {time.time()-start:.1f}s")
        else:
            logging.info("MMProj skipped (non-vision model or disabled)")

        report["success"] = True
        return report

    except Exception as e:
        logging.error(e)
        return report

def save_report(report: Dict, path: str):
    with open(path, "w") as f:
        json.dump(report, f, indent=2)


In [ ]:
hf_model_id = "Qwen/Qwen2.5-1.5B-Instruct"    # can also be a local path
model_name = "Qwen2.5-1.5B-Instruct"
output_folder = "ggufs"

report = convert_model_to_gguf(
    source_model=hf_model_id,
    model_name=model_name,
    quant_methods=["q4_k_m", "q8_0"],
    output_dir=output_folder,
    convert_mmproj=True,         # generate MMProj if vision model
    enable_mmproj_q8=False,      # Q8 MMProj optional (this isnt recommended just use BF16 mmproj)
)


In [ ]:
print(json.dumps(report, indent=2))

In [ ]:
!ls -lh ggufs

# import in sparknlp

In [ ]:
# This is only to setup PySpark and Spark NLP on Colab
!wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
import sparknlp

spark = sparknlp.start()

print("Spark NLP version: ", sparknlp.version())
print("Apache Spark version: ", spark.version)

specify the name of the model you want in sparknlp here

In [ ]:
sparknlp_model_name = "qwen2.5_1.5b_instruct"

In [ ]:
from sparknlp.annotator import AutoGGUFModel

imported_model = (
    AutoGGUFModel.loadSavedModel("/content/ggufs/qwen2.5-1.5b-instruct.q4_k_m.gguf", spark)
    .setInputCols("document")
    .setOutputCol("completions")
    .write().overwrite().save(sparknlp_model_name)
)


In [ ]:
!ls -lh {sparknlp_model_name}

In [ ]:
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import AutoGGUFModel
from pyspark.ml import Pipeline

document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

loaded_model = (
    AutoGGUFModel.load(sparknlp_model_name)
    .setInputCols("document")
    .setOutputCol("completions")
    .setBatchSize(1)
    .setNCtx(2048)
    .setNPredict(500)
    .setNBatch(512)
    .setNUbatch(512)
    .setUseMlock(True)
    .setUseMmap(True)
    .setTemperature(0.1)
)

pipeline = Pipeline(stages=[
    document_assembler,
    loaded_model
])

data = spark.createDataFrame([["""
A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus.
Which of the following is the best treatment for this patient?
A: Ampicillin
B: Ceftriaxone
C: Ciprofloxacin
D: Doxycycline
E: Nitrofurantoin
"""]]).toDF("text")

model = pipeline.fit(data)
result = model.transform(data)

result.select("completions.result").show(truncate=False)
